# Tutorial 05 — Gasificador 0D: Control PID y Optimización

**Equipo:** Gasificador de lecho fijo · **Modo:** 0D (N=1) · Semibatch
**Prerrequisito:** Tutorial 04 — Señales BC (ramp, step, ctrl P, ctrl onoff)

| Bloque | Herramienta | Concepto nuevo |
|--------|-------------|---------------|
| **5A** | Closure PID en runner | Acción integral → elimina error estacionario del ctrl P |
| **5B** | `parametric_sweep` + `optimize_bc` | Paisaje del objetivo → optimización precisa 1D |
| **5C** | `parametric_sweep` 2D + `optimize_bc` | Paisaje 2D como mapa de calor → optimización multivariable |

## Introducción

### Bloque 5A — PID

El controlador proporcional del tutorial 04 (caso 4E) tiene **error estacionario**:
al llegar al equilibrio térmico, Ts se estabiliza en SP + error_ss en lugar de SP.

La acción integral `Ki × ∫error dt` corrige esto acumulando la discrepancia con el tiempo.

**Dónde vive el integrador:** la integral se implementa como closure en el notebook,
no dentro de `callable(t, snap)` en el RHS. Esto evita que el integridad acumule
durante la estimación del Jacobiano del BDF (donde el RHS se llama varias veces
al mismo `t` con perturbaciones del vector de estado).

```
_integral[0] acumula solo cuando dt = t - _t_prev[0] > 0 (primera llamada por paso)
Llamadas de Jacobiano al mismo t: dt = 0 → sin acumulación
```

### Bloques 5B y 5C — Optimización

`optimize_bc` usa `scipy.optimize.minimize` con diferencias finitas para el gradiente.
Cada evaluación de gradiente requiere 1-2 simulaciones adicionales por variable.

**Estrategia recomendada:**
1. Mapear el paisaje del objetivo con `parametric_sweep` (visión global rápida).
2. Usar `optimize_bc` para encontrar el óptimo preciso dentro de esa región.

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.io.fuels_reader                      import read_fueldb
from src.units.gasifier.config.boundary_c     import build_bc_config
from src.units.gasifier.config.thermal_bc     import build_thermal_bc_config
from src.units.gasifier.config.transport      import build_transport_config
from src.units.gasifier.config.gas_props      import build_gas_prop_config, GASIFIER_GAS_SPECIES
from src.units.gasifier.config.solid_props    import build_solid_prop_config
from src.units.gasifier.config.initial_c      import build_initial_c_config
from src.solvers.runner_gasifier              import run_step
from src.postprocessing.gasifier_balances     import check_balances, display_balances
from src.postprocessing.gasifier_plots        import plot_sweep_profiles, plot_sweep_metrics
from src.utils.optimization                   import parametric_sweep, optimize_bc

FUEL_PATH = os.path.join(ROOT, "materials", "fuels", "softwood_spruce.yaml")
GAS_DB    = os.path.join(ROOT, "materials", "fluids", "gasdb.txt")

fuel_config = read_fueldb(FUEL_PATH)
print(f"ROOT: {ROOT}")
print(f"Combustible: {fuel_config['description']}")

## 1. Setup — mismo reactor que tutorial 04

In [ ]:
# ── Geometría (igual que tutorial 04) ────────────────────────────────────────
nc      = 9
N       = 1
species = list(GASIFIER_GAS_SPECIES)
Di, Do  = 0.10, 0.114
e_wall  = (Do - Di) / 2
L       = 0.50
dz      = L / N
Ai      = 0.25 * np.pi * Di**2
Pi, Po  = np.pi * Di, np.pi * Do
epsi_r  = 0.60
dp0     = float(fuel_config["physical"]["dp_initial"])
rho_p   = float(fuel_config["physical"]["rho_particle"])
rho_char0 = fuel_config["pyrolysis_yields"]["char"] * rho_p * (1 - epsi_r)

k_wall, rho_wall, Cp_wall = 16.5, 7950.0, 510.0
MC_WB   = 0.165
P_OUT   = 1.01325   # [bar]
T_END   = 3600.0    # [s]  tiempo largo para ver el control
RTOL    = 1e-5
ATOL    = 1e-7
N_SEC   = 3

# ── Propiedades ───────────────────────────────────────────────────────────────
prop_gas  = build_gas_prop_config(fuel_config=fuel_config, mode="polynomial", db_path=GAS_DB)
solid_cfg = build_solid_prop_config(fuel_config)
gas_T_ref = float(np.min(np.asarray(prop_gas["Tref"])))
MW_arr    = np.asarray(prop_gas["MW"])
trans_cfg = build_transport_config(mode="constant", N=N, n_comp=nc, h_bed=80.0, h_wall=12.0)
bc_out    = build_bc_config(n_comp=nc, P_out_bar=P_OUT, Cv=0.5)

rho_bio_0 = rho_p * (1 - epsi_r)
rho_moi_0 = MC_WB / (1 - MC_WB) * rho_bio_0
y0        = np.zeros(nc); y0[species.index("N2")] = 1.0
init      = build_initial_c_config(
    P_init=P_OUT, Tg_init=300.0, Ts_init=300.0, y_init=y0,
    rho_biomass_init=rho_bio_0, rho_char_init=1e-6, rho_moisture_init=rho_moi_0,
    n_comp=nc, N=N, prop_gas=prop_gas, epsi_r=epsi_r, gas_T_ref=gas_T_ref,
)
sv0 = init["sv0"]

params_base = {
    "n_comp":nc, "N":N, "dz":dz, "Ai":Ai, "Di":Di, "Pi":Pi, "Po":Po,
    "prop_gas":prop_gas, "MW":MW_arr, "gas_T_ref":gas_T_ref,
    "bc_config":bc_out, "trans_config":trans_cfg,
    "energy":True, "epsi_r":epsi_r, "dp0":dp0, "rho_char0":rho_char0,
    "fuel_config":fuel_config, "solid_config":solid_cfg, "species":species,
}

def make_params(tbc):
    return {**params_base, "thermal_bc_config": tbc, "_cache": {}}

# Caso referencia (T_wall constante) — base de comparacion
tbc_ref = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=1073.15, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
)
print("Simulando referencia (T_wall=1073 K)...")
t_ref, _, g_ref = run_step(sv0=sv0, t_max=T_END, params=make_params(tbc_ref),
                            rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True)
print(f"  rho_bio_fin={g_ref._rho_solid_results[-1,0,0]:.2f} kg/m3  "
      f"Ts={g_ref._Ts_results[-1,0]-273.15:.0f} C")
print(f"sv0.shape={sv0.shape}  rho_bio_0={rho_bio_0:.1f} kg/m3")

---
## 5A — Controlador PID

### Por qué el ctrl P tiene error estacionario

Con el controlador proporcional del tutorial 04 (4E), el equilibrio térmico satisface:
```
T_wall_ss = bias + Kp × (SP − Ts_ss)
```
Como T_wall_ss ≠ SP en general, Ts_ss ≠ SP → queda un error permanente.

### La acción integral

```
T_wall(t) = bias + Kp × e(t) + Ki × ∫₀ᵗ e(τ) dτ    e(t) = SP − Ts_mean(t)
```

El integrador acumula el error histórico y empuja T_wall hasta que e → 0.

### Implementación: closure en el notebook (no en el RHS)

El integrando vive fuera del RHS como estado mutable (lista de un elemento).
Dentro del RHS, el BDF llama a `callable(t, snap)` múltiples veces al mismo `t`
para estimar el Jacobiano. Con `dt = t − t_prev`:
- Primera llamada: dt > 0 → acumula correctamente.
- Llamadas del Jacobiano al mismo t: dt = 0 → sin acumulación espuria.

In [ ]:
SP_Ts    = 700.0    # [K]  setpoint Ts_mean (igual que casos 4E/4F)
Kp_pid   = 3.0      # [K_wall / K_error]  igual que ctrl P del tutorial 04
Ki_pid   = 0.003    # [K_wall / (K_error * s)]  accion integral
T_bias   = 800.0    # [K]  punto de operacion nominal
T_min    = 300.0
T_max    = 1200.0
WINDUP   = 500.0    # [K*s]  limite anti-windup del integrador

# ── Estado del integrador (closure, fuera del RHS) ────────────────────────────
_integral = [0.0]
_t_prev   = [0.0]

def pid_T_wall(t, snap):
    e      = SP_Ts - snap.get("Ts_mean", SP_Ts)
    dt     = max(0.0, float(t) - _t_prev[0])
    _integral[0] = float(np.clip(_integral[0] + e * dt, -WINDUP, WINDUP))
    _t_prev[0]   = float(t)
    return float(np.clip(T_bias + Kp_pid * e + Ki_pid * _integral[0], T_min, T_max))

# Resetear integrador antes de la simulacion
_integral[0] = 0.0
_t_prev[0]   = 0.0

tbc_pid = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=pid_T_wall, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
)

print(f"Simulando PID (SP={SP_Ts} K, Kp={Kp_pid}, Ki={Ki_pid})...")
t_pid, _, g_pid = run_step(
    sv0=sv0, t_max=T_END, params=make_params(tbc_pid),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
Ts_fin_pid = float(g_pid._Ts_results[-1, 0])
print(f"  Ts_final = {Ts_fin_pid-273.15:.1f} C  "
      f"(error_SS = {abs(Ts_fin_pid - SP_Ts):.1f} K)")

### 5A — Comparación: ctrl P vs PID (mismo SP = 700 K)

In [ ]:
# Ctrl proporcional del tutorial 04 (caso 4E) para comparacion
from src.control.controllers import proportional
ctrl_P = proportional(
    setpoint=SP_Ts, gain=Kp_pid,
    channel_in=lambda snap: snap.get("Ts_mean", 300.0),
    output_bias=T_bias, output_min=T_min, output_max=T_max,
)
tbc_P = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=ctrl_P, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
)
print("Simulando ctrl P (mismo Kp que PID, sin integral)...")
t_P, _, g_P = run_step(
    sv0=sv0, t_max=T_END, params=make_params(tbc_P),
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
Ts_fin_P = float(g_P._Ts_results[-1, 0])
print(f"  Ts_final = {Ts_fin_P-273.15:.1f} C  "
      f"(error_SS = {abs(Ts_fin_P - SP_Ts):.1f} K)")

# Reconstruir T_wall efectiva del ctrl P
Ts_h_P   = g_P._Ts_results[:, 0]
Tw_eff_P = np.clip(T_bias + Kp_pid * (SP_Ts - Ts_h_P), T_min, T_max)

# T_wall efectiva del PID: reconstruir desde Ts usando el mismo calculo
Ts_h_pid = g_pid._Ts_results[:, 0]
# Integral aproximada post-hoc (trapezoidal)
e_pid     = SP_Ts - Ts_h_pid
integ_ph  = np.zeros_like(e_pid)
for k in range(1, len(t_pid)):
    dt_k = t_pid[k] - t_pid[k-1]
    integ_ph[k] = np.clip(integ_ph[k-1] + e_pid[k-1] * dt_k, -WINDUP, WINDUP)
Tw_eff_pid = np.clip(T_bias + Kp_pid * e_pid + Ki_pid * integ_ph, T_min, T_max)

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
fig.suptitle("5A — Ctrl P vs PID  |  SP = 700 K  (0D, Cv=0.5)", fontsize=11)

axes[0].plot(t_pid/60, Tw_eff_pid, color='steelblue',  lw=2, label='PID')
axes[0].plot(t_P/60,   Tw_eff_P,   color='darkorange',  lw=2, label='ctrl P')
axes[0].axhline(T_bias, ls=':', color='gray', alpha=0.5, label='bias')
axes[0].set(xlabel="t [min]", ylabel="T_wall efectiva [K]", title="T_wall aplicada")
axes[0].legend(fontsize=8); axes[0].grid(True)

axes[1].plot(t_pid/60, Ts_h_pid-273.15, color='steelblue',  lw=2, label='PID')
axes[1].plot(t_P/60,   Ts_h_P-273.15,   color='darkorange',  lw=2, label='ctrl P')
axes[1].axhline(SP_Ts-273.15, color='r', ls='--', label=f'SP={SP_Ts-273.15:.0f} C')
axes[1].set(xlabel="t [min]", ylabel="Ts [C]", title="Temperatura solido")
axes[1].legend(fontsize=8); axes[1].grid(True)

axes[2].plot(t_pid/60, g_pid._rho_solid_results[:,0,0], color='steelblue',  lw=2, label='PID')
axes[2].plot(t_P/60,   g_P._rho_solid_results[:,0,0],   color='darkorange',  lw=2, label='ctrl P')
axes[2].plot(t_ref/60, g_ref._rho_solid_results[:,0,0], color='k', lw=1.5, ls='--', alpha=0.4, label='ref')
axes[2].set(xlabel="t [min]", ylabel="rho_bio [kg/m3_bed]", title="Biomasa restante")
axes[2].legend(fontsize=8); axes[2].grid(True)

axes[3].plot(t_pid/60, Ki_pid * integ_ph, color='steelblue', lw=2)
axes[3].axhline(0, color='gray', ls=':', alpha=0.5)
axes[3].set(xlabel="t [min]", ylabel="Ki x integral [K]",
            title="Contribucion integral del PID")
axes[3].grid(True)

plt.tight_layout(); plt.show()

# Tabla comparativa
bio0 = float(g_ref._rho_solid_results[0, 0, 0])
comp = {
    "ctrl": ["ctrl P", "PID (Ki={})".format(Ki_pid)],
    "Ts_fin [C]":   [round(Ts_fin_P-273.15,1),   round(Ts_fin_pid-273.15,1)],
    "error_SS [K]": [round(abs(Ts_fin_P-SP_Ts),1), round(abs(Ts_fin_pid-SP_Ts),1)],
    "conv_bio [%]": [round(100*(1-g_P._rho_solid_results[-1,0,0]/bio0),1),
                     round(100*(1-g_pid._rho_solid_results[-1,0,0]/bio0),1)],
}
display(pd.DataFrame(comp).set_index("ctrl"))

In [ ]:
bal_pid = check_balances(g_pid, make_params(tbc_pid), verbose=False)
display_balances(bal_pid)

---
## 5B — `optimize_bc`: T_wall óptima para maximizar conversión

**Objetivo:** encontrar la temperatura de pared constante que maximiza la conversión
de biomasa al final de la simulación (t=600 s).

**Estrategia en dos pasos:**
1. **Paisaje:** `parametric_sweep` sobre 15 valores de T_wall para visualizar el objetivo.
2. **Óptimo preciso:** `optimize_bc` con L-BFGS-B partiendo de la región de mayor conversión.

Ambos usan tolerancias holgadas (rtol=1e-3) para reducir el tiempo de cómputo del benchmark.
Para resultados de producción, usar rtol=1e-5.

In [ ]:
T_MAX_OPT = 600.0    # [s]  ventana de optimizacion
RTOL_OPT  = 1e-3     # tolerancias holgadas para velocidad
ATOL_OPT  = 1e-5
N_SEC_OPT = 2

def run_opt(params):
    """run_fn para optimize_bc y parametric_sweep — lee params['T_wall_K']."""
    Tw  = float(params["T_wall_K"])
    tbc = build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=Tw, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
    )
    p   = {**params_base,
           "bc_config": build_bc_config(n_comp=nc, P_out_bar=P_OUT, Cv=0.5),
           "thermal_bc_config": tbc, "_cache": {}}
    t_arr, _, g = run_step(
        sv0=sv0, t_max=T_MAX_OPT, params=p,
        rtol=RTOL_OPT, atol=ATOL_OPT, n_sec=N_SEC_OPT,
        show_progress=bool(params.get("_show_progress", False)),
    )
    g._t = t_arr
    return g

def obj_conv(g):
    """Fraccion de biomasa convertida al final (escalar)."""
    return 1.0 - float(g._rho_solid_results[-1, 0, 0]) / rho_bio_0

def metrics_opt(g):
    """Metricas para el DataFrame del barrido."""
    return {
        "conv_bio":  round(obj_conv(g), 4),
        "Ts_fin_C":  round(float(g._Ts_results[-1, 0]) - 273.15, 1),
        "P_max_bar": round(float(g._P_results.max()), 4),
        "y_CO_fin":  round(float(g._y_results[-1, species.index("CO"), 0]), 4),
    }

print("run_fn y objetivos listos.")
print(f"T_MAX_OPT={T_MAX_OPT:.0f} s  rtol={RTOL_OPT}  n_sec={N_SEC_OPT}")

### 5B.1 — Paisaje del objetivo (parametric_sweep)

In [ ]:
T_wall_grid = np.linspace(600.0, 1300.0, 15).tolist()

print(f"Mapeando paisaje: {len(T_wall_grid)} puntos de T_wall...")
t0 = time.perf_counter()
df_land, res_land = parametric_sweep(
    base_params={},
    sweep_vars={"T_wall_K": T_wall_grid},
    run_fn=run_opt,
    objective_fn=metrics_opt,
    return_results=True,
    n_jobs=1,
    verbose=True,
    show_sim_progress=False,
)
print(f"  Tiempo: {time.perf_counter()-t0:.1f} s")
display(df_land[["T_wall_K", "conv_bio", "Ts_fin_C", "P_max_bar", "y_CO_fin"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(df_land["T_wall_K"], df_land["conv_bio"], "o-", color="steelblue", lw=2)
axes[0].set(xlabel="T_wall [K]", ylabel="conv_bio [-]",
            title=f"Conversion de biomasa a t={T_MAX_OPT:.0f} s")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_land["T_wall_K"], df_land["y_CO_fin"], "s-", color="forestgreen",
             lw=2, label="y_CO")
axes[1].plot(df_land["T_wall_K"], df_land["Ts_fin_C"]/1000, "^--", color="darkorange",
             lw=1.5, label="Ts_fin [kC]")
axes[1].set(xlabel="T_wall [K]", ylabel="y_CO / Ts_fin [kC]",
            title="Composicion CO y temperatura final del solido")
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

fig.suptitle(f"5B — Paisaje del objetivo  (T_MAX={T_MAX_OPT:.0f} s, rtol={RTOL_OPT})",
             fontweight="bold")
fig.tight_layout(); plt.show()

### 5B.2 — Optimización precisa con `optimize_bc`

In [ ]:
print("Optimizando T_wall (L-BFGS-B, bounds=[600, 1300] K)...")
print("Cada linea = una evaluacion de la funcion objetivo\n")

t0 = time.perf_counter()
opt_b = optimize_bc(
    base_params={},
    decision_vars={"T_wall_K": (600.0, 1300.0)},
    run_fn=run_opt,
    objective_fn=lambda g: -obj_conv(g),   # negativo: minimizar = maximizar conv_bio
    method="L-BFGS-B",
    options={"maxiter": 30, "ftol": 1e-4},
    verbose=True,
)
t_opt_b = time.perf_counter() - t0
print(f"\nTiempo total: {t_opt_b:.1f} s  |  Evaluaciones: {opt_b['n_eval']}")

T_wall_opt = opt_b["x_opt"]["T_wall_K"]
conv_opt   = -opt_b["f_opt"]
print(f"\nT_wall optima: {T_wall_opt:.1f} K ({T_wall_opt-273.15:.1f} C)")
print(f"conv_bio maxima: {conv_opt:.4f}  ({conv_opt*100:.1f} %)")

In [ ]:
# Validacion del optimo con tolerancias ajustadas
print(f"Validando T_wall={T_wall_opt:.1f} K con rtol={RTOL}...")
tbc_opt_b = build_thermal_bc_config(
    mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
    T_wall=T_wall_opt, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
)
t_opt_b_val, _, g_opt_b = run_step(
    sv0=sv0, t_max=T_MAX_OPT,
    params={**params_base,
            "bc_config": build_bc_config(n_comp=nc, P_out_bar=P_OUT, Cv=0.5),
            "thermal_bc_config": tbc_opt_b, "_cache": {}},
    rtol=RTOL, atol=ATOL, n_sec=N_SEC, show_progress=True,
)
conv_val = obj_conv(g_opt_b)
print(f"  conv_bio (rtol={RTOL}): {conv_val:.4f}  ({conv_val*100:.1f} %)")

# Grafica: paisaje + trayectoria del optimizador + resultado validado
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_land["T_wall_K"], df_land["conv_bio"], "o-",
        color="steelblue", lw=2, label="Paisaje (rtol=1e-3)")
ax.axvline(T_wall_opt, color="r", ls="--", lw=2,
           label=f"T_wall optima = {T_wall_opt:.1f} K")
ax.scatter([T_wall_opt], [conv_val], color="r", s=100, zorder=5,
           label=f"Validado (rtol=1e-5): {conv_val*100:.1f} %")
ax.set(xlabel="T_wall [K]", ylabel="conv_bio [-]",
       title=f"Paisaje y optimo — T_MAX={T_MAX_OPT:.0f} s")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
fig.tight_layout(); plt.show()

---
## 5C — Paisaje 2D y optimización multivariable: T_wall + dp0

Se añade el diámetro de partícula `dp0` como segunda variable de decisión.
Partículas más pequeñas → mayor área específica → cinética de char más rápida (SCM).

**Paisaje 2D:** `parametric_sweep` cartesiano (5×4 = 20 casos) como mapa de calor.
**Óptimo 2D:** `optimize_bc` encuentra el punto exacto.

In [ ]:
def run_opt_2d(params):
    """run_fn para barrido 2D — lee T_wall_K y dp0 de params."""
    Tw   = float(params["T_wall_K"])
    dp   = float(params["dp0"])
    tbc  = build_thermal_bc_config(
        mode="fixed_twall", Di=Di, Do=Do, e_wall=e_wall,
        T_wall=Tw, k_wall=k_wall, rho_wall=rho_wall, Cp_wall=Cp_wall,
    )
    p    = {**params_base,
            "bc_config": build_bc_config(n_comp=nc, P_out_bar=P_OUT, Cv=0.5),
            "thermal_bc_config": tbc, "dp0": dp, "_cache": {}}
    t_arr, _, g = run_step(
        sv0=sv0, t_max=T_MAX_OPT, params=p,
        rtol=RTOL_OPT, atol=ATOL_OPT, n_sec=N_SEC_OPT,
        show_progress=bool(params.get("_show_progress", False)),
    )
    g._t = t_arr
    return g

Tw_grid  = [700.0, 850.0, 1000.0, 1073.15, 1200.0]
dp_grid  = [0.005, 0.010, 0.015, 0.020]

print(f"Paisaje 2D: {len(Tw_grid)}×{len(dp_grid)} = {len(Tw_grid)*len(dp_grid)} casos...")
t0 = time.perf_counter()
df_2d, _ = parametric_sweep(
    base_params={},
    sweep_vars={"T_wall_K": Tw_grid, "dp0": dp_grid},
    run_fn=run_opt_2d,
    objective_fn=metrics_opt,
    return_results=False,
    n_jobs=1,
    verbose=True,
    show_sim_progress=False,
)
print(f"  Tiempo: {time.perf_counter()-t0:.1f} s")

# Mapa de calor
pivot = df_2d.pivot(index="T_wall_K", columns="dp0", values="conv_bio")
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.pcolormesh(pivot.columns * 1000, pivot.index - 273.15,
                   pivot.values, cmap="viridis", shading="auto")
plt.colorbar(im, ax=ax, label="conv_bio [-]")
ax.set(xlabel="dp0 [mm]", ylabel="T_wall [C]",
       title=f"Conversion de biomasa — paisaje 2D (t={T_MAX_OPT:.0f} s)")
ax.grid(True, alpha=0.2, color="white")
fig.tight_layout(); plt.show()
display(pivot.round(3))

### 5C.2 — Óptimo 2D con `optimize_bc`

In [ ]:
print("Optimizando T_wall + dp0 (L-BFGS-B)...\n")

t0 = time.perf_counter()
opt_c = optimize_bc(
    base_params={},
    decision_vars={
        "T_wall_K": (600.0, 1300.0),
        "dp0":      (0.003, 0.025),
    },
    run_fn=run_opt_2d,
    objective_fn=lambda g: -obj_conv(g),
    method="L-BFGS-B",
    options={"maxiter": 40, "ftol": 1e-4},
    verbose=True,
)
t_opt_c = time.perf_counter() - t0
print(f"\nTiempo: {t_opt_c:.1f} s  |  Evaluaciones: {opt_c['n_eval']}")
print(f"T_wall optima: {opt_c['x_opt']['T_wall_K']:.1f} K  "
      f"({opt_c['x_opt']['T_wall_K']-273.15:.1f} C)")
print(f"dp0 optimo:   {opt_c['x_opt']['dp0']*1000:.2f} mm")
print(f"conv_bio max: {-opt_c['f_opt']:.4f}  ({-opt_c['f_opt']*100:.1f} %)")

# Marcar el optimo en el mapa de calor
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.pcolormesh(pivot.columns * 1000, pivot.index - 273.15,
                   pivot.values, cmap="viridis", shading="auto")
plt.colorbar(im, ax=ax, label="conv_bio [-]")
ax.scatter([opt_c["x_opt"]["dp0"]*1000],
           [opt_c["x_opt"]["T_wall_K"]-273.15],
           color="red", s=150, marker="*", zorder=5,
           label=f"optimo ({opt_c['x_opt']['T_wall_K']-273.15:.0f} C, "
                 f"{opt_c['x_opt']['dp0']*1000:.1f} mm)")
ax.set(xlabel="dp0 [mm]", ylabel="T_wall [C]",
       title="Paisaje 2D con optimo marcado")
ax.legend(fontsize=9); ax.grid(True, alpha=0.2, color="white")
fig.tight_layout(); plt.show()

---
## Conclusiones

### 5A — PID vs ctrl P

| Aspecto | Ctrl P | PID (Kp + Ki) |
|---------|--------|---------------|
| Error estacionario | Permanente (≠ 0) | Eliminado por integrador |
| Comportamiento transitorio | Más rápido al SP | Más lento al inicio (integrador parte de 0) |
| Riesgo | Ninguno | Windup si el sistema no puede alcanzar SP → usar anti-windup |
| Cuándo usar | Cuando se acepta offset | Cuando la precisión en SS es crítica |

El integrador se implementa como closure en el notebook. Para producción, moverlo
al runner entre sub-intervalos de `n_sec` para garantizar que no acumula durante
la estimación del Jacobiano del BDF.

### 5B — optimize_bc 1D

- `parametric_sweep` mapea el paisaje → identifica la región del óptimo sin coste extra.
- `optimize_bc` converge en esa región con pocas evaluaciones (gradiente numérico).
- Validar siempre el óptimo con tolerancias ajustadas (rtol=1e-5) después de optimizar
  con tolerancias holgadas (rtol=1e-3).

### 5C — optimize_bc 2D

- El producto cartesiano de `parametric_sweep` genera el mapa de calor 2D.
- `optimize_bc` multivariable encuentra el punto exacto con L-BFGS-B.
- Coste computacional: ~2×n_vars evaluaciones por iteración + gradiente numérico.
- Para espacios de más de 3-4 variables, considerar métodos de muestreo adaptativo.

### Patrón recomendado para optimización en ProSimNet

```
1. parametric_sweep (n_jobs=4)  → mapa del paisaje → identificar región de óptimo
2. optimize_bc (x0 = región)   → óptimo preciso
3. run_step con rtol=1e-5       → validar el óptimo con física de alta fidelidad
4. check_balances               → confirmar coherencia física del óptimo
```